# Clinical text embeddings: worked solution

Worked solution for the exercise in the lesson **[Clinical text embeddings](https://ramkumarmd.com/medical-ai/clinical-nlp/clinical-text-embeddings)** (*Clinical NLP* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](03-clinical-text-embeddings.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Text embeddings turn notes into vectors so that notes with similar meaning are close together, which lets a classifier recognise a problem described in words it was never trained on. On synthetic notes, an embedding-based classifier recognised foot problems in new wording far better than word counts did. But the same classifier gave 'denies tingling in both feet' the same score as 'reports tingling in both feet': embeddings capture the topic more than the assertion. Clinical use needs negation handling and validation on real local notes.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q sentence-transformers

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Embeddings as features

Synthetic notes; train on one set of phrases, test on paraphrases. TF-IDF baseline.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
openers = ["Seen in diabetes clinic.", "Annual review.", "Follow-up visit.", "Telephone consultation."]
foot_train = ["tingling in both feet", "numbness in the toes", "foot ulcer on the heel",
              "burning pain in the feet at night", "reduced sensation on monofilament testing"]
other_train = ["cough for one week", "headache since Monday", "medication review only",
               "blurred vision when reading", "knee pain after walking"]
# Test notes describe the same problems in words the training notes never use
foot_test = ["pins and needles in her soles", "cannot feel the floor properly",
             "sore on the sole that will not heal", "feet feel like walking on cotton wool",
             "wound under the big toe"]
other_test = ["chest infection improving", "migraine episodes", "routine prescription check",
              "eyes tired with screen use", "hip stiffness in the morning"]

def make_notes(phrases_pos, phrases_neg, n):
    notes, labels = [], []
    for _ in range(n):
        label = rng.integers(0, 2)
        phrase = rng.choice(phrases_pos if label else phrases_neg)
        notes.append(f"{rng.choice(openers)} Patient reports {phrase}. HbA1c {rng.uniform(6, 10):.1f}%.")
        labels.append(label)
    return notes, np.array(labels)

train_notes, y_train = make_notes(foot_train, other_train, 200)
test_notes, y_test = make_notes(foot_test, other_test, 200)

tfidf = TfidfVectorizer().fit(train_notes)
clf = LogisticRegression(max_iter=1000).fit(tfidf.transform(train_notes), y_train)
print(f"TF-IDF + logistic regression:     test AUROC "
      f"{roc_auc_score(y_test, clf.predict_proba(tfidf.transform(test_notes))[:, 1]):.3f}")

# TF-IDF + logistic regression:     test AUROC 0.825

### Embeddings recognise paraphrase

Continuing from the previous block: the same classifier on sentence embeddings.

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
E_train = encoder.encode(train_notes, normalize_embeddings=True)
E_test = encoder.encode(test_notes, normalize_embeddings=True)
clf_e = LogisticRegression(max_iter=1000).fit(E_train, y_train)
print(f"Sentence embeddings + logistic regression: test AUROC "
      f"{roc_auc_score(y_test, clf_e.predict_proba(E_test)[:, 1]):.3f}")

# Sentence embeddings + logistic regression: test AUROC 1.000

### Embeddings can ignore negation

Continuing from the previous block: three probe notes, including two that say there is no problem.

In [ ]:
probes = ["Annual review. Patient reports tingling in both feet. HbA1c 7.2%.",
          "Annual review. Patient denies tingling in both feet. HbA1c 7.2%.",
          "Annual review. No numbness, no ulcers, feet examined and normal. HbA1c 7.2%."]
for text, p in zip(probes, clf_e.predict_proba(encoder.encode(probes, normalize_embeddings=True))[:, 1]):
    print(f"P(foot problem) {p:.2f}  {text}")

# P(foot problem) 0.90  Annual review. Patient reports tingling in both feet. HbA1c 7.2%.
# P(foot problem) 0.90  Annual review. Patient denies tingling in both feet. HbA1c 7.2%.
# P(foot problem) 0.76  Annual review. No numbness, no ulcers, feet examined and normal. HbA1c 7.2%.

## Exercise

**Goal:** Build and stress-test embedding-based note classifiers.

**Data:** Synthetic notes generated in the notebook (Created for this course; no real patient data)

### Step 1

Reproduce the TF-IDF and embedding results and the negation probes.

In [ ]:
p_tfidf = clf.predict_proba(tfidf.transform(test_notes))[:, 1]
p_emb = clf_e.predict_proba(E_test)[:, 1]
print(f"TF-IDF AUROC {roc_auc_score(y_test, p_tfidf):.3f}; embeddings AUROC {roc_auc_score(y_test, p_emb):.3f}")

def probe(model_predict, label):
    print(label)
    for text, p in zip(probes, model_predict(probes)):
        print(f"   P(foot problem) {p:.2f}  {text}")

probe(lambda t: clf_e.predict_proba(encoder.encode(t, normalize_embeddings=True))[:, 1], "Embedding model:")

### Step 2

Add negated training notes ('denies…', 'no…') labelled as no problem, retrain, and re-run the probes.

In [ ]:
negated_phrases = ["no tingling in the feet", "denies numbness in the toes", "no foot ulcers",
                   "feet examined, sensation normal", "no burning pain in the feet"]
rng_neg = np.random.default_rng(1)
neg_notes = [f"{rng_neg.choice(openers)} Patient reports {rng_neg.choice(negated_phrases)}. HbA1c {rng_neg.uniform(6, 10):.1f}%."
             for _ in range(80)]
train_notes_2 = train_notes + neg_notes
y_train_2 = np.concatenate([y_train, np.zeros(len(neg_notes), dtype=int)])
clf_e2 = LogisticRegression(max_iter=1000).fit(encoder.encode(train_notes_2, normalize_embeddings=True), y_train_2)

probe(lambda t: clf_e2.predict_proba(encoder.encode(t, normalize_embeddings=True))[:, 1], "Retrained with negated notes:")
print(f"Paraphrase test AUROC after retraining {roc_auc_score(y_test, clf_e2.predict_proba(E_test)[:, 1]):.3f}")

Adding negated examples labelled "no problem" lowers the negated probes (0.90 to 0.51, and 0.76 to 0.26), but at a
cost: the genuine "reports tingling" probe falls to 0.60 and the paraphrase test AUROC drops from 1.000 to 0.886.
The embeddings place "no tingling in the feet" close to "tingling in the feet", so the classifier can only lower
confidence for anything foot-related. Real negation comes in many forms, and teaching it through examples alone needs
far more data than this.

### Step 3

Apply the NegEx-style rule from the first lesson to remove negated sentences before embedding, and compare.

In [ ]:
import re

NEGATION_CUES = r"\b(no|denies|denied|without|negative for|absent)\b"

def drop_negated_sentences(text):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return " ".join(s for s in sentences if not re.search(NEGATION_CUES, s.lower()))

filtered_probes = [drop_negated_sentences(p) for p in probes]
for original, filtered, p in zip(probes, filtered_probes,
                                 clf_e.predict_proba(encoder.encode(filtered_probes, normalize_embeddings=True))[:, 1]):
    print(f"P(foot problem) {p:.2f}  {filtered!r}  (from {original!r})")
filtered_test = [drop_negated_sentences(t) for t in test_notes]
print(f"Paraphrase test AUROC with negated sentences removed "
      f"{roc_auc_score(y_test, clf_e.predict_proba(encoder.encode(filtered_test, normalize_embeddings=True))[:, 1]):.3f}")

Removing negated sentences before embedding fixes the probes without retraining: only "Annual review. HbA1c 7.2%."
remains, which carries no foot information. It is a blunt tool: a sentence such as "no ulcers but numbness in both
feet" would be dropped entirely, losing the positive finding. Combining a negation step with negated training
examples is more robust than either alone.

### Step 4

Optional: replace the general model with a clinical encoder (for example Bio_ClinicalBERT with mean pooling) and compare on the paraphrase test.

In [ ]:
# Optional: a clinical encoder (Bio_ClinicalBERT, about 430 MB) with mean pooling, compared on the paraphrase test.
import torch
from transformers import AutoModel, AutoTokenizer

tok = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
bert = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT").eval()

def mean_pooled(texts, batch=32):
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            enc = tok(texts[i:i + batch], padding=True, truncation=True, max_length=64, return_tensors="pt")
            hidden = bert(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1)
            out.append(((hidden * mask).sum(1) / mask.sum(1)).numpy())   # average over real tokens only
    return np.vstack(out)

B_train, B_test = mean_pooled(train_notes), mean_pooled(test_notes)
clf_b = LogisticRegression(max_iter=2000).fit(B_train, y_train)
print(f"Bio_ClinicalBERT mean-pooled + logistic regression: paraphrase test AUROC "
      f"{roc_auc_score(y_test, clf_b.predict_proba(B_test)[:, 1]):.3f}")
print(f"all-MiniLM-L6-v2 sentence embeddings (for comparison):     "
      f"{roc_auc_score(y_test, clf_e.predict_proba(E_test)[:, 1]):.3f}")

Bio_ClinicalBERT was pretrained on clinical notes (MIMIC-III), but as a masked-language model, not to produce
sentence embeddings; simply averaging its token vectors often gives weaker sentence-level similarity than a model
trained for that purpose. Domain pretraining helps most when the encoder is fine-tuned on the task, or trained
specifically for sentence similarity, so compare rather than assume.

---

Lesson: [https://ramkumarmd.com/medical-ai/clinical-nlp/clinical-text-embeddings](https://ramkumarmd.com/medical-ai/clinical-nlp/clinical-text-embeddings). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.